In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from movebench.study import Study
from movebench.ssf import grouped_nll, iSSFDataset, get_issf_channel_info, MOVEMENT_FEATURES

In [3]:
# Load a study
study = Study.load('5a280544-06db-4c85-9713-842547972edd')
print(study)

Study('5a280544-06db-4c85-9713-842547972edd')


In [4]:
# Create iSSF dataset
issf_ds = study.issf_dataset(split='train')
print(f"iSSF dataset: {len(issf_ds)} samples")
print(f"Implementation: {type(issf_ds).__module__}.{type(issf_ds).__name__}")

# Load one sample
step_pixels, valid_pixels, transform, crs = issf_ds[0]
print(f"Step pixels shape: {step_pixels.shape}")

# Show channel layout
info = get_issf_channel_info()
print(f"\nChannel layout:")
print(f"  Env features:   channels {info['env'][0]}-{info['env'][1]-1} ({info['n_env']} channels)")
print(f"  sl_:            channel {info['sl_'][0]}")
print(f"  log_sl_:        channel {info['log_sl_'][0]}")
print(f"  cos_ta_:        channel {info['cos_ta_'][0]}")
print(f"  Interactions:   channels {info['interactions'][0]}-{info['interactions'][1]-1} ({info['n_interactions']} channels)")
print(f"  Total:          {info['total']} channels")

/data/vision/beery/scratch/justin/movebench/movebench/utils.py:181: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valid_step_next = df['valid_step'].shift(-1).fillna(False).astype(bool)


iSSF dataset: 1513 samples
Implementation: movebench.ssf.iSSFDataset
Step pixels shape: torch.Size([319, 101])

Channel layout:
  Env features:   channels 0-78 (79 channels)
  sl_:            channel 79
  log_sl_:        channel 80
  cos_ta_:        channel 81
  Interactions:   channels 82-318 (237 channels)
  Total:          319 channels


In [5]:
# Inspect movement features for the true endpoint (last column)
print("Movement features for true endpoint (index 100):")
print(f"  sl_     = {step_pixels[info['sl_'][0], -1].item():.2f} m")
print(f"  log_sl_ = {step_pixels[info['log_sl_'][0], -1].item():.4f}")
print(f"  cos_ta_ = {step_pixels[info['cos_ta_'][0], -1].item():.4f}")

print(f"\nMovement features for first pseudo-candidate (index 0):")
print(f"  sl_     = {step_pixels[info['sl_'][0], 0].item():.2f} m")
print(f"  log_sl_ = {step_pixels[info['log_sl_'][0], 0].item():.4f}")
print(f"  cos_ta_ = {step_pixels[info['cos_ta_'][0], 0].item():.4f}")

Movement features for true endpoint (index 100):
  sl_     = 3.35 m
  log_sl_ = 1.4906
  cos_ta_ = 1.3038

Movement features for first pseudo-candidate (index 0):
  sl_     = -0.15 m
  log_sl_ = 0.6847
  cos_ta_ = -1.2612


In [6]:
# Check validity
move_start, move_end = info['movement']
print(f"Movement validity (should be all True): {valid_pixels[move_start:move_end].all().item()}")
print(f"Overall valid ratio: {valid_pixels.float().mean().item():.2%}")

Movement validity (should be all True): True
Overall valid ratio: 76.21%


---
## Train iSSF Model

Train a conditional logistic regression model using the iSSFDataset.
The model learns coefficients for environmental features, movement features,
and their interactions.

In [7]:
# Extract features from iSSFDataset

def extract_ssf_features(dataset, num_workers=8):
    """Extract all features from an SSFDataset into tensors."""
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=num_workers)
    
    all_xs = []
    all_masks = []
    
    for step_pixels, valid_pixels, _, _ in tqdm(loader, desc="Extracting features"):
        all_xs.append(step_pixels)
        all_masks.append(valid_pixels)
    
    # Stack: (N, C, num_candidates+1)
    X = torch.cat(all_xs, dim=0)
    masks = torch.cat(all_masks, dim=0)
    
    # Reshape to (N * (num_candidates+1), C) for regression
    N, C, K = X.shape
    X = X.permute(0, 2, 1).reshape(-1, C)
    masks = masks.permute(0, 2, 1).reshape(-1, C)
    
    # Labels: last candidate (index K-1) is the true endpoint
    y = torch.zeros(N * K, dtype=torch.long)
    y[K-1::K] = 1  # Every K-th element starting from K-1
    
    # Group indices for conditional logistic regression
    starts = torch.arange(0, N * K, K)
    ends = torch.arange(K, N * K + 1, K)
    
    return X, masks, y, starts, ends

# Extract train features
train_issf = study.issf_dataset(split='train')

X_tr, masks_tr, y_tr, starts_tr, ends_tr = extract_ssf_features(train_issf)

print(f"Train features: {X_tr.shape}")
print(f"Train groups: {len(starts_tr)}")

/data/vision/beery/scratch/justin/movebench/movebench/utils.py:181: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valid_step_next = df['valid_step'].shift(-1).fillna(False).astype(bool)
Extracting features: 100%|██████████| 1513/1513 [02:03<00:00, 12.25it/s]


Train features: torch.Size([152813, 319])
Train groups: 1513


In [8]:
# Clean features: fill invalid values with 0 (approx mean after normalization)
def clean_features(X, masks):
    X_clean = torch.where(masks, X, torch.tensor(0.0))
    X_clean = torch.nan_to_num(X_clean, nan=0.0, posinf=0.0, neginf=0.0)
    return X_clean

X_tr_clean = clean_features(X_tr, masks_tr)

print(f"Features per sample: {X_tr_clean.shape[1]}")

Features per sample: 319


In [13]:
# Train conditional logistic regression
num_features = X_tr_clean.shape[1]
beta = torch.zeros(num_features, dtype=torch.float32, requires_grad=True)
opt = torch.optim.Adam([beta], 
                       lr=1e-2, 
                    #    weight_decay=1e-3
                       )

for step in range(5000):
    opt.zero_grad()
    loss = grouped_nll(beta, X_tr_clean, y_tr, starts_tr, ends_tr)
    loss.backward()
    opt.step()
    
    if step % 100 == 0:
        with torch.no_grad():
            nll_tr = grouped_nll(beta, X_tr_clean, y_tr, starts_tr, ends_tr).item()
            print(f"Step {step:4d} | Train NLL: {nll_tr:.4f}")

Step    0 | Train NLL: 4.5107
Step  100 | Train NLL: 4.2042
Step  200 | Train NLL: 4.1882
Step  300 | Train NLL: 4.1808
Step  400 | Train NLL: 4.1764
Step  500 | Train NLL: 4.1736
Step  600 | Train NLL: 4.1716
Step  700 | Train NLL: 4.1701
Step  800 | Train NLL: 4.1691
Step  900 | Train NLL: 4.1682
Step 1000 | Train NLL: 4.1675
Step 1100 | Train NLL: 4.1675
Step 1200 | Train NLL: 4.1667
Step 1300 | Train NLL: 4.1662
Step 1400 | Train NLL: 4.1659
Step 1500 | Train NLL: 4.1656
Step 1600 | Train NLL: 4.1654
Step 1700 | Train NLL: 4.1652
Step 1800 | Train NLL: 4.1650
Step 1900 | Train NLL: 4.1649
Step 2000 | Train NLL: 4.1647
Step 2100 | Train NLL: 4.1646
Step 2200 | Train NLL: 4.1645
Step 2300 | Train NLL: 4.1646
Step 2400 | Train NLL: 4.1643
Step 2500 | Train NLL: 4.1642
Step 2600 | Train NLL: 4.1642
Step 2700 | Train NLL: 4.1645
Step 2800 | Train NLL: 4.1640
Step 2900 | Train NLL: 4.1639
Step 3000 | Train NLL: 4.1640
Step 3100 | Train NLL: 4.1639
Step 3200 | Train NLL: 4.1638
Step 3300 

In [15]:
# Inspect learned coefficients
from movebench.config import get_channel_indices, LAYERS

# Build channel names matching the iSSF layout
indices = get_channel_indices()
env_channel_names = []
for name, (start, end) in indices.items():
    layer = LAYERS[name]
    if layer.categorical:
        for i in range(layer.num_classes):
            env_channel_names.append(f"{name}_class{i}")
    else:
        for band in layer.bands:
            env_channel_names.append(f"{name}_{band}")

# Movement feature names
move_names = list(MOVEMENT_FEATURES)

# Interaction names
interact_names = []
for mf in MOVEMENT_FEATURES:
    for env_name in env_channel_names:
        interact_names.append(f"{mf}:{env_name}")

channel_names = env_channel_names + move_names + interact_names

beta_np = beta.detach().numpy()
sorted_idx = abs(beta_np).argsort()[::-1]

print("Top 15 coefficients by magnitude:\n")
for i in sorted_idx[:15]:
    print(f"{channel_names[i]:45s} {beta_np[i]:+.4f}")

print("\n--- Movement feature coefficients ---")
info = get_issf_channel_info()
move_start, move_end = info['movement']
for i in range(move_start, move_end):
    print(f"{channel_names[i]:45s} {beta_np[i]:+.4f}")

Top 15 coefficients by magnitude:

bioclim_bio_4                                 +6.3151
bioclim_bio_10                                -4.7832
bioclim_bio_14                                +4.3935
era5_mean_sea_level_pressure                  +4.2202
modis_lc_class16                              -3.2591
bioclim_bio_12                                +2.9986
era5_land_surface_net_solar_radiation         +2.9790
log_sl_:bioclim_bio_14                        -2.7105
sl_:bioclim_bio_9                             +2.4397
log_sl_:modis_lc_class17                      +2.4307
bioclim_bio_15                                +2.2382
sl_:bioclim_bio_2                             -2.2226
log_sl_:modis_lc_class16                      -2.1848
sl_:bioclim_bio_3                             +2.1618
bioclim_bio_2                                 -2.1613

--- Movement feature coefficients ---
sl_                                           -0.3908
log_sl_                                       -0.1728
cos_ta_ 

## Test Set Evaluation

In [11]:
import time

# Time test set extraction
test_issf = study.issf_dataset(split='test')
print(f"Test dataset: {len(test_issf)} samples")

start = time.time()
X_te, masks_te, y_te, starts_te, ends_te = extract_ssf_features(test_issf)
elapsed = time.time() - start

print(f"Test extraction time: {elapsed:.1f}s")
print(f"Test features: {X_te.shape}")

/data/vision/beery/scratch/justin/movebench/movebench/utils.py:181: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valid_step_next = df['valid_step'].shift(-1).fillna(False).astype(bool)


Test dataset: 2155 samples


Extracting features: 100%|██████████| 2155/2155 [15:21<00:00,  2.34it/s]


Test extraction time: 921.7s
Test features: torch.Size([217655, 319])


In [16]:
# Evaluate on test set
X_te_clean = clean_features(X_te, masks_te)

with torch.no_grad():
    nll_te = grouped_nll(beta, X_te_clean, y_te, starts_te, ends_te).item()
    print(f"Test NLL: {nll_te:.4f}")

Test NLL: 4.7355
